In [1]:
import sys, os
# Adjust depth so Python can find config.py at the project root.
# notebooks live 2 levels deep (notebooks/<phase_folder>/notebook.ipynb)
sys.path.insert(0, os.path.abspath('../..'))
from config import *

import os, re, pickle, warnings
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from tqdm import tqdm
warnings.filterwarnings('ignore')
print('Imports and config loaded.')


Imports and config loaded.


# Phase 4 — Feature Extraction & Engineering

## Overview

This phase converts the preprocessed text from Phase 3 into numerical feature matrices that machine learning models can train on. Two types of features are built and combined:

| Feature type | Description | Count |
|---|---|---|
| TF-IDF | Weighted word/bigram frequencies from lemmatised text | 5,000 |
| Handcrafted | Structural, positional and pharmacological signal features | 24 |
| **Total** | Combined sparse feature matrix | **5,024** |

**Pipeline position:**
```
Phase 1: Parse → Phase 2: EDA → Phase 3: Preprocess → [Phase 4: Features] → Phase 5: Train
```

**Input:** `train_processed.csv`, `test_processed.csv`  
**Output:** `train_features.npz`, `test_features.npz`, `y_train.npy`, `y_test.npy`, `tfidf_vectorizer.pkl`, `label_encoder.pkl`

---

## Bugs fixed vs original `feature_extraction.py`

| # | Severity | Bug | Fix |
|---|---|---|---|
| 1 | Critical | `get_drug_positions()` searches for `@drug1$` — token doesn't exist | Search for `drug1` in `sentence_cleaned` |
| 2 | Critical | `get_words_between_drugs()` searches for `@drug1$` — always returns -1 | Regex updated to match `drug1`/`drug2` |
| 3 | Important | Per-type keyword counts from Phase 3 (`kw_mechanism` etc) not used | All 4 keyword type columns now used as features |
| 4 | Important | `drug1_type`/`drug2_type` preserved by Phase 3 but ignored here | Encoded as 6 binary features — EDA showed brand+group has 50.4% interaction rate |
| 5 | Important | `is_self_pair` flag from Phase 3 not used | Added as binary feature |
| 6 | Important | `corpus_source` not used | Added as binary feature (DrugBank vs MedLine) |
| 7 | Minor | No column validation at startup | Validates all required columns exist before processing |
| 8 | Minor | `iterrows()` in handcrafted feature loop | Replaced with `itertuples()` — 3–5x faster |

## 1. Imports & Dependencies

- `scipy.sparse` — efficient sparse matrix operations for TF-IDF
- `sklearn.feature_extraction.text.TfidfVectorizer` — TF-IDF implementation
- `sklearn.preprocessing.LabelEncoder` — converts string labels to integers
- `pickle` — saves fitted vectoriser and encoder for reuse in Phase 5
- `tqdm` — progress bar

**Install if needed:**
```bash
pip install scikit-learn scipy numpy pandas tqdm
```

## 2. Configuration

**`MAX_TFIDF_FEATURES` explained:**  
The EDA found ~5,929 unique tokens in the corpus. Setting `MAX_TFIDF_FEATURES=5000` captures the 5,000 most informative tokens. You could also set it to `None` to use all tokens — the difference is small but 5,000 keeps the matrix manageable.

**`FILTER_SELF_PAIRS` explained:**  
Phase 3 flagged 1,990 training rows where drug1 == drug2. These are annotation artefacts — all but 6 are labelled `false`. Setting this to `True` removes them from training, which gives the model cleaner signal. Set to `False` to keep them.

In [2]:
# ── All paths come from config.py ─────────────────────────────────
# Inputs    : TRAIN_PROCESSED, TEST_PROCESSED
# Outputs   : TRAIN_FEATURES_NPZ, TEST_FEATURES_NPZ, Y_TRAIN_NPY, Y_TEST_NPY
#             TFIDF_VECTORISER_PKL, LABEL_ENCODER_PKL, FEATURE_NAMES_PKL
# Parameters: MAX_TFIDF_FEATURES, FILTER_SELF_PAIRS

for path in [TRAIN_PROCESSED, TEST_PROCESSED]:
    status = 'FOUND' if os.path.exists(path) else 'NOT FOUND — run Phase 3 first'
    print(f'{path}  =>  {status}')


/Users/carinadesouza/Documents/Projects/Python/Drug Drug Interaction NLP project/data/processed/train_processed.csv  =>  FOUND
/Users/carinadesouza/Documents/Projects/Python/Drug Drug Interaction NLP project/data/processed/test_processed.csv  =>  FOUND


## 3. Load & Validate Data

Before extracting any features, we validate that all required columns are present. This catches the common mistake of passing the raw CSV (from Phase 1) instead of the processed CSV (from Phase 3).

**Required columns from Phase 3:**
- `lemmas` — lemmatised text for TF-IDF
- `sentence_cleaned` — for drug position and keyword detection
- `keyword_count`, `kw_mechanism`, `kw_effect`, `kw_advise`, `kw_general` — keyword features
- `is_self_pair` — self-pair flag
- `drug1_type`, `drug2_type` — entity type features
- `corpus_source` — DrugBank vs MedLine
- `interaction_type` — target label

In [3]:
REQUIRED_COLS = [
    'lemmas', 'sentence_cleaned', 'sentence_masked',
    'keyword_count', 'kw_mechanism', 'kw_effect', 'kw_advise', 'kw_general',
    'is_self_pair', 'drug1_type', 'drug2_type', 'corpus_source',
    'interaction_type',
]

print('Loading data...')
train_df = pd.read_csv(TRAIN_PROCESSED)
test_df = pd.read_csv(TEST_PROCESSED)

# Column validation
print('\nValidating required columns...')
for name, df in [('TRAIN', train_df), ('TEST', test_df)]:
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(
            f'{name} is missing columns: {missing}\n'
            f'Make sure you are using the Phase 3 PROCESSED CSV, not the raw Phase 1 CSV.'
        )
    print(f'{name}: all {len(REQUIRED_COLS)} required columns present.')

print(f'\nTrain rows: {len(train_df):,}')
print(f'Test rows: {len(test_df):,}')
print(f'Train columns: {list(train_df.columns)}')

Loading data...

Validating required columns...
TRAIN: all 13 required columns present.
TEST: all 13 required columns present.

Train rows: 27,792
Test rows: 6,657
Train columns: ['sentence_id', 'drug1_text', 'drug1_type', 'drug1_norm', 'drug2_text', 'drug2_type', 'drug2_norm', 'ddi', 'interaction_type', 'corpus_source', 'offset_valid', 'source_file', 'sentence_original', 'sentence_masked', 'sentence_cleaned', 'tokens', 'tokens_no_stop', 'lemmas', 'is_self_pair', 'keywords_found', 'keyword_count', 'kw_mechanism', 'kw_effect', 'kw_advise', 'kw_general']


## 4. Optional: Filter Self-Pairs from Training

Phase 3 flagged 1,990 training rows where drug1 and drug2 are the same drug. From the EDA:
- 1,984 of 1,990 are labelled `false` (99.7%)
- Only 6 are labelled as actual interactions (annotation errors)
- They produce `DRUG1 ... DRUG1` masked sentences, not `DRUG1 ... DRUG2`

These rows are noise. Filtering them makes the training set cleaner. The model trained without them should learn more reliable patterns since every training example will have two distinct drugs.

This is controlled by `FILTER_SELF_PAIRS` in the configuration cell above.

In [4]:
# FIX: ensure boolean comparison works for both bool and string 'True'/'False'
if FILTER_SELF_PAIRS and 'is_self_pair' in train_df.columns:
    before = len(train_df)
    # Use ~ (not) instead of == False to handle both bool and string columns
    train_df = train_df[~train_df['is_self_pair'].astype(bool)].reset_index(drop=True)
    print(f'Self-pair filter applied: {before:,} -> {len(train_df):,} rows ({before-len(train_df)} removed)')
else:
    self_count = train_df['is_self_pair'].sum() if 'is_self_pair' in train_df.columns else 'unknown'
    print(f'Self-pair filter OFF. {self_count} self-pairs retained in training data.')

print(f'\nFinal training set: {len(train_df):,} rows')
print(f'Test set (unchanged): {len(test_df):,} rows')

Self-pair filter applied: 27,792 -> 25,802 rows (1990 removed)

Final training set: 25,802 rows
Test set (unchanged): 6,657 rows


## 5. TF-IDF Feature Matrix

TF-IDF (Term Frequency–Inverse Document Frequency) converts text into a numerical matrix where each column represents a word or bigram, and each value represents how important that term is in that document relative to the whole corpus.

### Key settings explained

| Parameter | Value | Reason |
|---|---|---|
| `max_features` | 5,000 | Keeps most informative terms, limits matrix size |
| `sublinear_tf` | True | Dampens very frequent words (log scaling) — EDA showed Zipf distribution |
| `ngram_range` | (1,2) | Captures bigrams like 'increase risk', 'inhibit metabolism' |
| `min_df` | 2 | Ignores terms appearing in fewer than 2 documents — removes noise |
| `token_pattern` | `\w[\w-]+` | Preserves hyphenated drug terms like 'anti-coagulant' |

### Critical: fit on train only, transform both
The vectoriser is **fitted on training data only**, then used to transform both train and test. Fitting on test data would cause **data leakage** — the model would indirectly see test set vocabulary during training.

In [5]:
print('Building TF-IDF features...')
print('Fitting vectoriser on TRAINING data only (prevents data leakage)')

vectoriser = TfidfVectorizer(
    max_features=MAX_TFIDF_FEATURES,
    sublinear_tf=True, # log(1 + tf) — dampens very frequent words
    ngram_range=(1, 2), # unigrams + bigrams
    min_df=2, # must appear in at least 2 documents
    token_pattern=r'(?u)\b\w[\w\-]+\b', # preserves hyphenated terms
)

train_tfidf = vectoriser.fit_transform(train_df['lemmas'].fillna(''))
test_tfidf = vectoriser.transform(test_df['lemmas'].fillna(''))

print(f'Train TF-IDF shape: {train_tfidf.shape}')
print(f'Test TF-IDF shape: {test_tfidf.shape}')
print(f'Vocabulary size: {len(vectoriser.vocabulary_):,}')
print(f'Matrix sparsity: {1 - train_tfidf.nnz / (train_tfidf.shape[0]*train_tfidf.shape[1]):.3%}')

# Show top terms for a sample row
print('\nTop 10 TF-IDF terms for first training row:')
feature_names = vectoriser.get_feature_names_out()
row0 = train_tfidf[0].toarray()[0]
top_idx = row0.argsort()[-10:][::-1]
for idx in top_idx:
    if row0[idx] > 0:
        print(f'{feature_names[idx]:<30} {row0[idx]:.4f}')

Building TF-IDF features...
Fitting vectoriser on TRAINING data only (prevents data leakage)
Train TF-IDF shape: (25802, 5000)
Test TF-IDF shape: (6657, 5000)
Vocabulary size: 5,000
Matrix sparsity: 99.047%

Top 10 TF-IDF terms for first training row:
measure                        0.3273
laboratory test                0.3158
laboratory                     0.3088
total                          0.3054
week                           0.2975
29                             0.2889
drug2 concentration            0.2777
prior                          0.2666
monitor                        0.2619
test                           0.2467


## 6. Handcrafted Features

These 24 features capture structural and pharmacological properties that TF-IDF cannot represent. They are divided into four groups:

### Group A — Drug position features (4 features)
Where the two drugs appear in the sentence and how far apart they are. EDA showed `effect` pairs have drugs 9.5 words apart on average vs `int` pairs at 16 words — distance is a genuine signal.

**Bug fixed:** Original code searched for `@drug1$` which doesn't exist. Fixed to search for `drug1`/`drug2` in `sentence_cleaned`.

### Group B — Keyword type counts (4 features)
Per-type keyword counts from Phase 3: `kw_mechanism`, `kw_effect`, `kw_advise`, `kw_general`. These were computed in Phase 3 but ignored in the original feature extraction — now used directly.

### Group C — Entity type features (7 features)
Drug entity type encoding from EDA findings:
- `brand+group` pairs have a **50.4% interaction rate** vs `drug+drug` at 11.4%
- `brand+brand` pairs have only **1.7% interaction rate**
- Entity type is one of the strongest predictors identified in EDA

### Group D — Data quality & corpus features (3 features)
- `is_self_pair` — 1,990 noisy training rows where drug1==drug2
- `is_drugbank` — corpus source (DrugBank has 14.6% interaction rate, MedLine 13.0%)
- `sentence_length` — longer sentences tend to be `false` (EDA finding)

In [6]:
def get_drug_positions(sentence_cleaned: str):
    """
    Find normalised positions (0.0-1.0) of DRUG1 and DRUG2 in the cleaned sentence.

    Bug fixed: original searched for '@drug1$' which doesn't exist after Phase 3
    preprocessing. sentence_cleaned contains 'drug1' and 'drug2' (lowercase, no
    special characters — they were stripped by clean_text()).

    Returns (pos1, pos2) where 0.0 = start of sentence, 1.0 = end.
    Default is 0.5 (middle) if token not found.
    """
    words = sentence_cleaned.split()
    total = len(words) if words else 1
    pos1, pos2 = 0.5, 0.5

    for i, word in enumerate(words):
        if word == 'drug1':   # exact match — token is a standalone word
            pos1 = i / total
        if word == 'drug2':
            pos2 = i / total

    return pos1, pos2


def get_words_between_drugs(sentence_cleaned: str) -> int:
    """
    Count words between drug1 and drug2 tokens in the cleaned sentence.

    Bug fixed: original regex searched for '@drug1$..@drug2$' — neither exists.
    Fixed to match 'drug1' and 'drug2' (what Phase 3 actually produces).

    Returns -1 if both tokens not found (sentinel value).
    """
    pattern = re.search(
        r'\bdrug1\b(.*?)\bdrug2\b|\bdrug2\b(.*?)\bdrug1\b',
        sentence_cleaned
    )
    if pattern:
        between = pattern.group(1) or pattern.group(2) or ''
        return len(between.split())
    return -1


# Verify the fix
print('Drug position function tests:')
sample = 'laboratory tests response to drug1 should be monitored by measuring drug2 concentrations'
p1, p2 = get_drug_positions(sample)
dist = get_words_between_drugs(sample)
print(f'Sentence: "{sample[:70]}..."')
print(f'DRUG1 position: {p1:.3f}  (should NOT be 0.5 — default means broken)')
print(f'DRUG2 position: {p2:.3f}')
print(f'Words between: {dist} (should NOT be -1 — that means broken)')
assert p1 != 0.5 or p2 != 0.5, 'Position detection still broken!'
assert dist != -1, 'Words-between detection still broken!'
print('Position detection working correctly.')

Drug position function tests:
Sentence: "laboratory tests response to drug1 should be monitored by measuring dr..."
DRUG1 position: 0.333  (should NOT be 0.5 — default means broken)
DRUG2 position: 0.833
Words between: 5 (should NOT be -1 — that means broken)
Position detection working correctly.


In [7]:
# Drug entity type encoding
# Based on EDA: brand+group = 50.4% interaction rate, drug+drug = 11.4%
DRUG_TYPES = ['brand', 'group', 'drug_n']  # 'drug' is the baseline (not encoded)


def extract_handcrafted_features(df: pd.DataFrame) -> np.ndarray:
    """
    Extract 24 handcrafted features per drug pair.

    Feature groups:
        A (0-3) : Drug position — pos1, pos2, distance, words_between
        B (4-8) : Keyword counts — total, mechanism, effect, advise, general
        C (9-15) : Entity type encoding — drug1 brand/group/drug_n, drug2 brand/group/drug_n, same_type
        D (16-18): Data quality — is_self_pair, is_drugbank, sentence_length
        E (19-23): Keyword flags — negation, interact, contraindicated, monitor, toxicity/bleeding

    Optimisation: itertuples() instead of iterrows() — 3-5x faster.
    """
    NEGATION = {'not', 'no', 'without', 'neither', 'nor', 'never', 'none'}
    INTERACT = {'interact', 'interaction'}
    CONTRA   = {'contraindicated'}
    MONITOR  = {'monitor', 'caution', 'avoid'}
    DANGER   = {'toxicity', 'bleeding', 'adverse'}

    rows = []

    for row in tqdm(df.itertuples(index=False), total=len(df), desc='Handcrafted features'):
        cleaned = str(getattr(row, 'sentence_cleaned', '')).lower()
        words_set = set(cleaned.split())

        # Group A: drug position
        pos1, pos2 = get_drug_positions(cleaned)
        distance = abs(pos1 - pos2)
        words_between = get_words_between_drugs(cleaned)

        # Group B: keyword counts (from Phase 3 output)
        kw_total = int(getattr(row, 'keyword_count', 0))
        kw_mech = int(getattr(row, 'kw_mechanism', 0))
        kw_eff = int(getattr(row, 'kw_effect', 0))
        kw_adv = int(getattr(row, 'kw_advise', 0))
        kw_gen = int(getattr(row, 'kw_general', 0))

        # Group C: entity type encoding
        # EDA finding: brand+group has 50.4% interaction rate vs drug+drug 11.4%
        d1_type = str(getattr(row, 'drug1_type', 'drug')).lower()
        d2_type = str(getattr(row, 'drug2_type', 'drug')).lower()
        d1_brand = int(d1_type == 'brand')
        d1_group = int(d1_type == 'group')
        d1_drug_n = int(d1_type == 'drug_n')
        d2_brand = int(d2_type == 'brand')
        d2_group = int(d2_type == 'group')
        d2_drug_n = int(d2_type == 'drug_n')
        same_type = int(d1_type == d2_type)  # same entity type

        # Group D: data quality & corpus
        is_self = int(bool(getattr(row, 'is_self_pair', False)))
        is_dbank = int(str(getattr(row, 'corpus_source', '')).lower() == 'drugbank')
        sent_len = len(cleaned.split())

        # Group E: keyword presence flags
        has_neg = int(bool(words_set & NEGATION))
        has_interact = int(bool(words_set & INTERACT))
        has_contra = int(bool(words_set & CONTRA))
        has_monitor = int(bool(words_set & MONITOR))
        has_danger = int(bool(words_set & DANGER))

        rows.append([
            # A: position
            pos1, pos2, distance, words_between,
            # B: keyword counts
            kw_total, kw_mech, kw_eff, kw_adv, kw_gen,
            # C: entity types
            d1_brand, d1_group, d1_drug_n,
            d2_brand, d2_group, d2_drug_n,
            same_type,
            # D: quality & corpus
            is_self, is_dbank, sent_len,
            # E: keyword flags
            has_neg, has_interact, has_contra, has_monitor, has_danger,
        ])

    return np.array(rows, dtype=np.float32)


FEATURE_NAMES = [
    # A
    'drug1_position', 'drug2_position', 'position_distance', 'words_between',
    # B
    'kw_total', 'kw_mechanism', 'kw_effect', 'kw_advise', 'kw_general',
    # C
    'drug1_is_brand', 'drug1_is_group', 'drug1_is_drug_n',
    'drug2_is_brand', 'drug2_is_group', 'drug2_is_drug_n',
    'same_entity_type',
    # D
    'is_self_pair', 'is_drugbank', 'sentence_length',
    # E
    'has_negation', 'has_interact', 'has_contraindicated', 'has_monitor', 'has_danger',
]

print(f'Handcrafted feature count: {len(FEATURE_NAMES)}')

Handcrafted feature count: 24


## 7. Extract Handcrafted Features

In [8]:
print('Extracting handcrafted features...')
train_hc = extract_handcrafted_features(train_df)
test_hc = extract_handcrafted_features(test_df)

print(f'\nTrain handcrafted shape: {train_hc.shape}')
print(f'Test  handcrafted shape: {test_hc.shape}')

# Show feature summary as DataFrame for readability
hc_df = pd.DataFrame(train_hc, columns=FEATURE_NAMES)
print('\nHandcrafted feature statistics (training set):')
print(hc_df.describe().round(3).to_string())

Extracting handcrafted features...


Handcrafted features: 100%|██████████| 6657/6657 [00:00<00:00, 64145.88it/s]


Train handcrafted shape: (25802, 24)
Test  handcrafted shape: (6657, 24)

Handcrafted feature statistics (training set):
       drug1_position  drug2_position  position_distance  words_between   kw_total  kw_mechanism  kw_effect  kw_advise  kw_general  drug1_is_brand  drug1_is_group  drug1_is_drug_n  drug2_is_brand  drug2_is_group  drug2_is_drug_n  same_entity_type  is_self_pair  is_drugbank  sentence_length  has_negation  has_interact  has_contraindicated  has_monitor  has_danger
count       25802.000       25802.000          25802.000      25802.000  25802.000     25802.000  25802.000  25802.000   25802.000       25802.000       25802.000        25802.000       25802.000       25802.000        25802.000         25802.000       25802.0    25802.000        25802.000     25802.000     25802.000            25802.000    25802.000   25802.000
mean            0.413           0.677              0.301          9.587      2.246         0.935      0.146      0.237       0.927           0.102  

## 8. Feature Quality Check — Mean by Interaction Type

Before combining everything, verify the handcrafted features actually differ between classes. A feature that has the same value for all classes provides no signal and should be dropped.

Key things to look for:
- `has_negation` should be highest for `false` class (EDA showed 26.6%)
- `kw_mechanism` should be highest for `mechanism` class
- `drug1_is_brand` + `drug2_is_group` together signal high interaction probability
- `is_self_pair` should be nearly 0 for interaction classes

In [9]:
hc_df['interaction_type'] = train_df['interaction_type'].values

LABEL_ORDER = ['false', 'effect', 'mechanism', 'advise', 'int']
check_features = [
    'has_negation', 'kw_mechanism', 'kw_effect', 'kw_advise',
    'drug1_is_brand', 'drug1_is_group', 'is_self_pair', 'position_distance'
]

print('Mean feature value by interaction type:')
print('(higher values in non-false classes = feature is discriminative)\n')
pivot = hc_df.groupby('interaction_type')[check_features].mean().reindex(LABEL_ORDER)
print(pivot.round(3).to_string())

print('\nValidation:')
neg_false = pivot.loc['false', 'has_negation']
neg_effect = pivot.loc['effect','has_negation']
print(f'has_negation — false: {neg_false:.3f}  effect: {neg_effect:.3f}')
if neg_false > neg_effect:
    print('PASS — negation is higher in false class (matches EDA finding)')
else:
    print('WARN — negation pattern unexpected')

Mean feature value by interaction type:
(higher values in non-false classes = feature is discriminative)

                  has_negation  kw_mechanism  kw_effect  kw_advise  drug1_is_brand  drug1_is_group  is_self_pair  position_distance
interaction_type                                                                                                                   
false                    0.195         0.961      0.129      0.214           0.090           0.273           0.0              0.274
effect                   0.072         0.267      0.401      0.257           0.132           0.368           0.0              0.443
mechanism                0.075         1.724      0.116      0.210           0.156           0.132           0.0              0.477
advise                   0.312         0.481      0.142      0.857           0.283           0.174           0.0              0.419
int                      0.032         0.407      0.063      0.169           0.037           0.106    

## 9. Combine TF-IDF + Handcrafted Features

The TF-IDF matrix is **sparse** (mostly zeros — most words don't appear in most sentences). The handcrafted matrix is **dense** (24 values for every row).

They are combined using `scipy.sparse.hstack()` which keeps the TF-IDF part sparse for memory efficiency while appending the handcrafted columns. The result is a single feature matrix that the classifier sees.

In [10]:
print('Combining TF-IDF + handcrafted features...')

def combine_features(tfidf_matrix, handcrafted_matrix):
    """Horizontally stack sparse TF-IDF with dense handcrafted features."""
    hc_sparse = sparse.csr_matrix(handcrafted_matrix)
    return sparse.hstack([tfidf_matrix, hc_sparse], format='csr')

X_train = combine_features(train_tfidf, train_hc)
X_test  = combine_features(test_tfidf,  test_hc)

print(f'Final X_train shape : {X_train.shape}')
print(f'Final X_test  shape : {X_test.shape}')
print(f'TF-IDF features : {train_tfidf.shape[1]}')
print(f'Handcrafted features: {train_hc.shape[1]}')
print(f'Total features : {X_train.shape[1]}')
print(f'Matrix density : {X_train.nnz / (X_train.shape[0]*X_train.shape[1]):.4%}')

Combining TF-IDF + handcrafted features...
Final X_train shape : (25802, 5024)
Final X_test  shape : (6657, 5024)
TF-IDF features : 5000
Handcrafted features: 24
Total features : 5024
Matrix density : 1.1431%


## 10. Label Encoding

Converts string labels to integers for sklearn models:
```
advise=0  effect=1  false=2  int=3  mechanism=4
```

The encoder is **fitted on training labels only**, then used to transform test labels. This ensures consistency — the same integer always maps to the same class.

In [11]:
# NOTE: LabelEncoder sorts classes alphabetically:
# advise=0, effect=1, false=2, int=3, mechanism=4
# This is consistent with Phase 1 labels (verified: no unexpected labels)
print('Encoding labels...')
encoder = LabelEncoder()
y_train = encoder.fit_transform(train_df['interaction_type'])
y_test  = encoder.transform(test_df['interaction_type'])

print(f'Classes : {list(encoder.classes_)}')
print(f'Mapping : { {c: int(i) for i, c in enumerate(encoder.classes_)} }')
print(f'y_train shape: {y_train.shape}')
print(f'y_test  shape: {y_test.shape}')

print('\nClass counts in training labels:')
from collections import Counter
counts = Counter(y_train)
for cls_int, count in sorted(counts.items()):
    cls_name = encoder.classes_[cls_int]
    print(f'{cls_int} ({cls_name:<12}) : {count:,}')

Encoding labels...
Classes : ['advise', 'effect', 'false', 'int', 'mechanism']
Mapping : {'advise': 0, 'effect': 1, 'false': 2, 'int': 3, 'mechanism': 4}
y_train shape: (25802,)
y_test  shape: (6657,)

Class counts in training labels:
0 (advise      ) : 826
1 (effect      ) : 1,685
2 (false       ) : 21,787
3 (int         ) : 189
4 (mechanism   ) : 1,315


## 11. Save All Outputs

All outputs are saved to `data/features/` — the **single source of truth** for this phase.

| File | Contents | Used by |
|---|---|---|
| `train_features.npz` | Sparse feature matrix (25,802 × 5,024) | Phase 5 training |
| `test_features.npz` | Sparse feature matrix (6,657 × 5,024) | Phase 5 evaluation |
| `y_train.npy` | Encoded training labels | Phase 5 training |
| `y_test.npy` | Encoded test labels | Phase 5 evaluation |
| `tfidf_vectorizer.pkl` | Fitted TF-IDF vectoriser | Phase 6 inference on new sentences |
| `label_encoder.pkl` | Fitted label encoder | Phase 5 `encoder.inverse_transform()`, Phase 6 inference |
| `feature_names.pkl` | List of all 5,024 feature names | Phase 5 feature importance plot |

> **Phase 5 does NOT re-save these files.** It loads them directly from here.
> **Phase 6 inference loads `tfidf_vectorizer.pkl` and `label_encoder.pkl` directly from `data/features/`** to transform new sentences into the same feature space.


In [12]:
print(f'Saving feature matrices to: {str(FEATURES_MATRICES_DIR)}')
print(f'Saving artefacts      to:   {str(FEATURES_ARTEFACTS_DIR)}')

# Feature matrices
sparse.save_npz(TRAIN_FEATURES_NPZ, X_train)
sparse.save_npz(TEST_FEATURES_NPZ,  X_test)
print('train_features.npz saved')
print('test_features.npz  saved')

# Labels
import numpy as _np
_np.save(Y_TRAIN_NPY, y_train)
_np.save(Y_TEST_NPY,  y_test)
print('y_train.npy saved')
print('y_test.npy  saved')

# Fitted artefacts — single source of truth, Phase 5 and 6 load from here
with open(TFIDF_VECTORISER_PKL, 'wb') as f:
    pickle.dump(vectoriser, f)
print('tfidf_vectoriser.pkl saved')

with open(LABEL_ENCODER_PKL, 'wb') as f:
    pickle.dump(encoder, f)
print('label_encoder.pkl saved')

all_feature_names = list(vectoriser.get_feature_names_out()) + FEATURE_NAMES
with open(FEATURE_NAMES_PKL, 'wb') as f:
    pickle.dump(all_feature_names, f)
print('feature_names.pkl  saved')

print(f'\nAll files saved.')


Saving feature matrices to: /Users/carinadesouza/Documents/Projects/Python/Drug Drug Interaction NLP project/features/matrices
Saving artefacts      to:   /Users/carinadesouza/Documents/Projects/Python/Drug Drug Interaction NLP project/features/artefacts
train_features.npz saved
test_features.npz  saved
y_train.npy saved
y_test.npy  saved
tfidf_vectoriser.pkl saved
label_encoder.pkl saved
feature_names.pkl  saved

All files saved.


## 12. Final Feature Summary

Complete breakdown of everything that was built.

In [13]:
print('=' * 60)
print('PHASE 4 COMPLETE — FEATURE SUMMARY')
print('=' * 60)
print(f'Training samples : {X_train.shape[0]:,}')
print(f'Test samples : {X_test.shape[0]:,}')
print(f'Total features per sample : {X_train.shape[1]:,}')
print(f'TF-IDF features : {train_tfidf.shape[1]:,}')
print(f'Handcrafted features : {train_hc.shape[1]:,}')
print(f'Label mapping : { {c: int(i) for i, c in enumerate(encoder.classes_)} }')
print()
print('Handcrafted feature groups:')
groups = [
    ('A — Drug position  ', 'drug1_position, drug2_position, position_distance, words_between'),
    ('B — Keyword counts ', 'kw_total, kw_mechanism, kw_effect, kw_advise, kw_general'),
    ('C — Entity types   ', 'drug1/2 brand/group/drug_n flags, same_entity_type'),
    ('D — Quality/corpus ', 'is_self_pair, is_drugbank, sentence_length'),
    ('E — Keyword flags  ', 'has_negation, has_interact, has_contraindicated, has_monitor, has_danger'),
]
for group, features in groups:
    print(f'{group}: {features}')

print()
print('Files saved:')
for fname in sorted(os.listdir(str(FEATURES_MATRICES_DIR))):
    fpath = str(FEATURES_MATRICES_DIR) + "/"  # use paths from config: fname)
    size  = os.path.getsize(fpath) / 1024
    print(f'{fname:<30} {size:8.1f} KB')

print()
print('Next: Phase 5 — Model Training')

PHASE 4 COMPLETE — FEATURE SUMMARY
Training samples : 25,802
Test samples : 6,657
Total features per sample : 5,024
TF-IDF features : 5,000
Handcrafted features : 24
Label mapping : {'advise': 0, 'effect': 1, 'false': 2, 'int': 3, 'mechanism': 4}

Handcrafted feature groups:
A — Drug position  : drug1_position, drug2_position, position_distance, words_between
B — Keyword counts : kw_total, kw_mechanism, kw_effect, kw_advise, kw_general
C — Entity types   : drug1/2 brand/group/drug_n flags, same_entity_type
D — Quality/corpus : is_self_pair, is_drugbank, sentence_length
E — Keyword flags  : has_negation, has_interact, has_contraindicated, has_monitor, has_danger

Files saved:
test_features.npz                   0.2 KB
train_features.npz                  0.2 KB
y_test.npy                          0.2 KB
y_train.npy                         0.2 KB

Next: Phase 5 — Model Training


## 13. Complete Feature Reference

| Index | Feature name | Type | Description |
|---|---|---|---|
| 0 | `drug1_position` | float 0–1 | Where DRUG1 appears in sentence |
| 1 | `drug2_position` | float 0–1 | Where DRUG2 appears in sentence |
| 2 | `position_distance` | float 0–1 | Absolute distance between drugs |
| 3 | `words_between` | int | Words between DRUG1 and DRUG2 (-1 if not found) |
| 4 | `kw_total` | int | Total pharmacological keywords found |
| 5 | `kw_mechanism` | int | Mechanism-type keyword count |
| 6 | `kw_effect` | int | Effect-type keyword count |
| 7 | `kw_advise` | int | Advise-type keyword count |
| 8 | `kw_general` | int | General interaction keyword count |
| 9 | `drug1_is_brand` | 0/1 | Drug1 is a brand name |
| 10 | `drug1_is_group` | 0/1 | Drug1 is a drug class/group |
| 11 | `drug1_is_drug_n` | 0/1 | Drug1 is experimental |
| 12 | `drug2_is_brand` | 0/1 | Drug2 is a brand name |
| 13 | `drug2_is_group` | 0/1 | Drug2 is a drug class/group |
| 14 | `drug2_is_drug_n` | 0/1 | Drug2 is experimental |
| 15 | `same_entity_type` | 0/1 | Drug1 and Drug2 are the same entity type |
| 16 | `is_self_pair` | 0/1 | Drug1 text == Drug2 text |
| 17 | `is_drugbank` | 0/1 | Sentence comes from DrugBank corpus |
| 18 | `sentence_length` | int | Word count of cleaned sentence |
| 19 | `has_negation` | 0/1 | Sentence contains not/no/without/never/etc |
| 20 | `has_interact` | 0/1 | Sentence contains 'interact'/'interaction' |
| 21 | `has_contraindicated` | 0/1 | Sentence contains 'contraindicated' |
| 22 | `has_monitor` | 0/1 | Sentence contains 'monitor'/'caution'/'avoid' |
| 23 | `has_danger` | 0/1 | Sentence contains 'toxicity'/'bleeding'/'adverse' |

In [14]:
# ── Final cross-check against Phase 1 output counts ─────────────────────────
print('=== Phase 1 → Phase 4 Pipeline Integrity Check ===')
print(f'Expected train rows : 27,792 | Got: {X_train.shape[0]:,}')
print(f'Expected test rows  :  6,657 | Got: {X_test.shape[0]:,}')
print(f'Feature count       :        | Got: {X_train.shape[1]:,}')
assert X_train.shape[0] in [27792, 25802], f'Unexpected train rows: {X_train.shape[0]}'
assert X_test.shape[0] == 6657, f'Unexpected test rows: {X_test.shape[0]}'
print('\nAll counts verified. Ready for Phase 5 (Model Training).')

# Expected: 27792 (all rows) or 25802 (self-pairs filtered: 27792 - 1990)
expected = 25802 if FILTER_SELF_PAIRS else 27792
if X_train.shape[0] == expected:
    print(f'PASS — row count matches FILTER_SELF_PAIRS={FILTER_SELF_PAIRS} setting')
else:
    print(f'WARN — got {X_train.shape[0]} rows, expected {expected}')


=== Phase 1 → Phase 4 Pipeline Integrity Check ===
Expected train rows : 27,792 | Got: 25,802
Expected test rows  :  6,657 | Got: 6,657
Feature count       :        | Got: 5,024

All counts verified. Ready for Phase 5 (Model Training).
PASS — row count matches FILTER_SELF_PAIRS=True setting
